# GCP FinOps Report
**Author:** Jorge Rodriguez  
**Description:** Queries GCP Billing export in BigQuery, aggregates cost data, and produces an interactive HTML report for CIO/CFO leadership.  
**Run:** Twice a month — all dates auto-calculated from today.

In [1]:
# Uncomment to install dependencies
# !pip3 install google-cloud-bigquery db-dtypes plotly pandas
#
# Authenticate once via CLI:
# gcloud auth application-default login

In [2]:
# Run this cell once, then restart the kernel.
# Uses the kernel's own Python binary so packages install to the right environment.
'''
import sys
!{sys.executable} -m pip install --upgrade google-cloud-bigquery db-dtypes plotly pandas nbformat
'''
#
# Authenticate once via CLI:
# gcloud auth application-default login

'\nimport sys\n!{sys.executable} -m pip install --upgrade google-cloud-bigquery db-dtypes plotly pandas nbformat\n'

In [3]:
import datetime
from datetime import date
from dateutil.relativedelta import relativedelta

from google.cloud import bigquery
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"
print("Imports OK")

Imports OK


## Configuration
Dates are auto-calculated to the previous complete month. Override `reportYear` / `reportMonth` manually if needed.

In [4]:
# ── Auto-calculate previous month ──────────────────────────────────────────
today               = date.today()
first_of_this_month = today.replace(day=1)
first_of_prev_month = first_of_this_month - relativedelta(months=1)

reportYear   = first_of_prev_month.strftime('%Y')
reportMonth  = first_of_prev_month.strftime('%m')
reportStart  = first_of_prev_month.strftime('%Y-%m-%d')
reportEnd    = first_of_this_month.strftime('%Y-%m-%d')
invoice_month_param = f"{reportYear}{reportMonth}"

# ── Override here if needed ────────────────────────────────────────────────
# reportYear  = '2026'
# reportMonth = '05'
# reportStart = '2026-05-01'
# reportEnd   = '2026-06-01'
# invoice_month_param = f"{reportYear}{reportMonth}"

print(f"Report Period : {reportStart}  →  {reportEnd}")
print(f"Invoice Month : {invoice_month_param}")

Report Period : 2026-05-01  →  2026-06-01
Invoice Month : 202605


## BigQuery Connection & Table Discovery

In [5]:
client     = bigquery.Client(project='billing-query-371121')
project_id = client.project

datasets  = list(client.list_datasets())
dataset   = datasets[0]
dataset_id = f"{project_id}.{dataset.dataset_id}"

tables = list(client.list_tables(dataset_id))
print(f"Tables in {dataset_id}:")
for t in tables:
    print(f"  {t.table_id}")

# Use the standard v1 export (not the resource-level one)
table_id = next(
    (f"{t.project}.{t.dataset_id}.{t.table_id}" for t in tables if "resource" not in t.table_id),
    f"{tables[-1].project}.{tables[-1].dataset_id}.{tables[-1].table_id}"
)
print(f"\nUsing table: {table_id}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Tables in billing-query-371121.all_billing_data:
  gcp_billing_export_resource_v1_01C5AB_0209B1_81C892
  gcp_billing_export_v1_01C5AB_0209B1_81C892

Using table: billing-query-371121.all_billing_data.gcp_billing_export_v1_01C5AB_0209B1_81C892


## Monthly Aggregated Query
Aggregation is pushed into BigQuery SQL — returns thousands of rows instead of millions.

In [6]:
monthly_sql = f"""
SELECT
  invoice.month                        AS invoice_month,
  project.id                           AS project_id,
  project.name                         AS project_name,
  service.description                  AS service_description,
  location.region                      AS region,
  SUM(cost)                            AS total_cost,
  SUM(IFNULL(cost_at_list, cost))      AS list_cost,
  SUM(IFNULL(cost_at_list, cost)) - SUM(cost) AS total_credits
FROM `{table_id}`
WHERE invoice.month = @invoice_month
  AND cost > 0.001
GROUP BY 1, 2, 3, 4, 5
ORDER BY total_cost DESC
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("invoice_month", "STRING", invoice_month_param)
    ]
)

monthly_df = client.query(monthly_sql, job_config=job_config).to_dataframe()

total_month = monthly_df['total_cost'].sum()
print(f"Rows returned : {len(monthly_df):,}")
print(f"Total cost    : ${total_month:,.2f}")

Rows returned : 527
Total cost    : $511,368.47


## Historical Multi-Year Query
One SQL query replaces reading and concatenating dozens of 30 GB CSV files.

In [7]:
import time

# ── Years to include — add/remove years here ──────────────────────────────
YEARS = [2023, 2024, 2025, 2026]

current_year = int(reportYear)
prior_year   = current_year - 1

trend_rows = []   # collects monthly totals per year
proj_rows  = []   # collects top-30 project totals per year

for year in YEARS:
    month_start = f"{year}01"
    month_end   = f"{year}12"
    print(f"Fetching {year} ...", end="  ", flush=True)

    # ── Monthly totals for this year  (12 rows max) ───────────────────────
    sql_monthly = f"""
    SELECT
      {year}                                         AS invoice_year,
      CAST(SUBSTR(invoice.month, 5, 2) AS INT64)    AS invoice_month_num,
      ROUND(SUM(cost), 2)                            AS total_cost
    FROM `{table_id}`
    WHERE cost > 0
      AND invoice.month BETWEEN '{month_start}' AND '{month_end}'
    GROUP BY 2
    ORDER BY 2
    """
    df_m = client.query(sql_monthly).to_dataframe()
    trend_rows.append(df_m)

    # ── Top 30 projects for this year  (30 rows max) ──────────────────────
    sql_proj = f"""
    SELECT
      {year}                  AS invoice_year,
      project.name            AS project_name,
      ROUND(SUM(cost), 2)     AS total_cost
    FROM `{table_id}`
    WHERE cost > 0
      AND invoice.month BETWEEN '{month_start}' AND '{month_end}'
    GROUP BY 2
    ORDER BY total_cost DESC
    LIMIT 30
    """
    df_p = client.query(sql_proj).to_dataframe()
    proj_rows.append(df_p)

    print(f"✓  {len(df_m)} months | {len(df_p)} projects | ${df_m['total_cost'].sum():,.0f}")
    time.sleep(0.5)   # small pause between API calls

# ── Combine results ────────────────────────────────────────────────────────
monthly_totals_df = pd.concat(trend_rows, ignore_index=True)
proj_annual_df    = pd.concat(proj_rows,  ignore_index=True)
proj_annual_df['invoice_year'] = proj_annual_df['invoice_year'].astype(int)

# ── Build yearly totals from monthly data (no extra query needed) ──────────
yearly_total = (
    monthly_totals_df.groupby('invoice_year')['total_cost']
    .sum().reset_index()
    .rename(columns={'total_cost': 'annual_cost'})
    .sort_values('invoice_year')
)
yearly_total['yoy_pct'] = yearly_total['annual_cost'].pct_change() * 100
yearly_total['label'] = yearly_total.apply(
    lambda r: f"${r['annual_cost']:,.0f}" +
              (f"<br>({r['yoy_pct']:+.1f}% YoY)" if pd.notna(r['yoy_pct']) else ""),
    axis=1
)

print(f"\nDone — {len(monthly_totals_df)} trend rows | {len(proj_annual_df)} project rows")

Fetching 2023 ...  ✓  12 months | 30 projects | $1,335,354
Fetching 2024 ...  ✓  12 months | 30 projects | $1,371,611
Fetching 2025 ...  ✓  12 months | 30 projects | $2,223,846
Fetching 2026 ...  ✓  6 months | 30 projects | $2,053,582

Done — 42 trend rows | 120 project rows


## Visualizations

In [8]:
# ── Top 20 Projects this month ─────────────────────────────────────────────
proj_monthly = (
    monthly_df.groupby('project_name')['total_cost']
    .sum().reset_index()
    .sort_values('total_cost', ascending=True)
    .tail(20)
)

fig_proj = px.bar(
    proj_monthly,
    x='total_cost', y='project_name',
    orientation='h',
    title=f"Top 20 Projects by Cost — {reportStart} to {reportEnd}",
    labels={'total_cost': 'Total Cost (USD)', 'project_name': 'Project'},
    text=proj_monthly['total_cost'].apply(lambda x: f"${x:,.0f}"),
    color='total_cost', color_continuous_scale='Blues',
)
fig_proj.update_traces(textposition='outside')
fig_proj.update_layout(
    height=650, coloraxis_showscale=False,
    xaxis_tickformat='$,.0f', margin=dict(l=200),
)
fig_proj.show()

In [9]:
# ── Top 15 Services this month ─────────────────────────────────────────────
svc_monthly = (
    monthly_df.groupby('service_description')['total_cost']
    .sum().reset_index()
    .sort_values('total_cost', ascending=True)
    .tail(15)
)

fig_svc = px.bar(
    svc_monthly,
    x='total_cost', y='service_description',
    orientation='h',
    title=f"Top 15 Services by Cost — {reportStart} to {reportEnd}",
    labels={'total_cost': 'Total Cost (USD)', 'service_description': 'Service'},
    text=svc_monthly['total_cost'].apply(lambda x: f"${x:,.0f}"),
    color='total_cost', color_continuous_scale='Greens',
)
fig_svc.update_traces(textposition='outside')
fig_svc.update_layout(
    height=550, coloraxis_showscale=False,
    xaxis_tickformat='$,.0f', margin=dict(l=220),
)
fig_svc.show()

In [10]:
# ── Treemap: service cost proportion (executives love this) ────────────────
svc_treemap = (
    monthly_df.groupby('service_description')['total_cost']
    .sum().reset_index()
    .query('total_cost > 100')
)

fig_tree = px.treemap(
    svc_treemap,
    path=['service_description'],
    values='total_cost',
    color='total_cost',
    color_continuous_scale='RdYlGn_r',
    title=f"Cost Distribution by Service (services > $100) — {reportStart} to {reportEnd}",
)
fig_tree.update_traces(
    texttemplate="<b>%{label}</b><br>$%{value:,.0f}",
    hovertemplate="<b>%{label}</b><br>Cost: $%{value:,.2f}<extra></extra>",
)
fig_tree.update_layout(height=520, coloraxis_showscale=False)
fig_tree.show()

In [12]:
# ── Monthly spend trend by year (line chart shows trajectory clearly) ──────
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

trend_data = monthly_totals_df.copy()
trend_data['invoice_year'] = trend_data['invoice_year'].astype(str)

fig_trend = px.line(
    trend_data,
    x='invoice_month_num', y='total_cost',
    color='invoice_year',
    markers=True,
    title="GCP Monthly Spend Trend by Year",
    labels={'invoice_month_num': 'Month', 'total_cost': 'Total Cost (USD)', 'invoice_year': 'Year'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_trend.update_layout(
    height=500,
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), ticktext=MONTH_LABELS),
    yaxis_tickformat='$,.0f',
    hovermode='x unified',
)
fig_trend.show()

In [14]:
# ── Annual totals with YoY % change annotation ────────────────────────────
# yearly_total is already computed in the history cell — just plot it.
fig_year = px.bar(
    yearly_total,
    x='invoice_year', y='annual_cost',
    text='label',
    title="GCP Annual Spend with YoY Change",
    labels={'invoice_year': 'Year', 'annual_cost': 'Annual Cost (USD)'},
    color='annual_cost', color_continuous_scale='Blues',
)
fig_year.update_traces(textposition='outside', textfont_size=12)
fig_year.update_layout(
    height=500, coloraxis_showscale=False,
    yaxis_tickformat='$,.0f',
    xaxis=dict(type='category'),
)
fig_year.show()

In [16]:
# ── Executive Dashboard: YoY project comparison ────────────────────────────
current_year = int(reportYear)
prior_year   = current_year - 1

proj_yoy = (
    proj_annual_df[proj_annual_df['invoice_year'].isin([prior_year, current_year])]
    .groupby(['invoice_year', 'project_name'])['total_cost']
    .sum().reset_index()
)

proj_pivot = proj_yoy.pivot_table(
    index='project_name', columns='invoice_year',
    values='total_cost', fill_value=0
).reset_index()
proj_pivot.columns.name = None

for yr in [prior_year, current_year]:
    if yr not in proj_pivot.columns:
        proj_pivot[yr] = 0.0

proj_pivot = proj_pivot.rename(columns={prior_year: 'prior_year', current_year: 'current_year'})
proj_pivot['cost_change'] = proj_pivot['current_year'] - proj_pivot['prior_year']
proj_pivot['pct_change']  = (
    proj_pivot['cost_change'] / proj_pivot['prior_year'].replace(0, float('nan'))
) * 100
proj_pivot['category'] = proj_pivot.apply(
    lambda r: 'New' if r['prior_year'] == 0
    else ('Increased' if r['cost_change'] > 0 else 'Reduced'),
    axis=1
)

top10      = proj_pivot.nlargest(10, 'current_year')
increases  = proj_pivot[proj_pivot['cost_change'] > 0].nlargest(10, 'cost_change')
reductions = proj_pivot[proj_pivot['cost_change'] < 0].nsmallest(10, 'cost_change')

fig_exec = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f"Top 10 Projects: {prior_year} vs {current_year}",
        "Project Category Distribution",
        "Top 10 Cost Increases (YoY)",
        "Top 10 Cost Reductions (YoY)",
    ),
    specs=[[{"type": "bar"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]],
)

fig_exec.add_trace(go.Bar(
    name=str(prior_year), x=top10['project_name'],
    y=top10['prior_year'], marker_color='#3498db',
), row=1, col=1)
fig_exec.add_trace(go.Bar(
    name=str(current_year), x=top10['project_name'],
    y=top10['current_year'], marker_color='#e74c3c',
), row=1, col=1)

cat_counts = proj_pivot['category'].value_counts()
fig_exec.add_trace(go.Pie(
    labels=cat_counts.index, values=cat_counts.values,
    marker_colors=['#e74c3c', '#2ecc71', '#3498db'],
    textinfo='percent+label', showlegend=False,
), row=1, col=2)

fig_exec.add_trace(go.Bar(
    x=increases['project_name'], y=increases['cost_change'],
    marker_color='#e74c3c', showlegend=False,
    text=increases['cost_change'].apply(lambda x: f"+${x:,.0f}"),
    textposition='outside',
), row=2, col=1)

fig_exec.add_trace(go.Bar(
    x=reductions['project_name'], y=reductions['cost_change'].abs(),
    marker_color='#2ecc71', showlegend=False,
    text=reductions['cost_change'].apply(lambda x: f"-${abs(x):,.0f}"),
    textposition='outside',
), row=2, col=2)

fig_exec.update_layout(
    height=900,
    title_text=f"Executive Dashboard: GCP Cost Analysis {prior_year} vs {current_year}",
    title_font_size=20,
    barmode='group',
)
fig_exec.update_yaxes(tickformat='$,.0f')
fig_exec.update_xaxes(tickangle=-45)
fig_exec.show()

## Executive Summary (Text)

In [17]:
total_this_month  = monthly_df['total_cost'].sum()
total_current_ytd = yearly_total[yearly_total['invoice_year'] == current_year]['annual_cost'].values
total_prior_full  = yearly_total[yearly_total['invoice_year'] == prior_year]['annual_cost'].values
total_current_ytd = total_current_ytd[0] if len(total_current_ytd) else 0
total_prior_full  = total_prior_full[0]  if len(total_prior_full)  else 0
yoy_change = total_current_ytd - total_prior_full
yoy_pct    = (yoy_change / total_prior_full * 100) if total_prior_full > 0 else float('nan')

print("=" * 65)
print(f"  GCP FinOps Executive Summary")
print(f"  Report Period : {reportStart}  →  {reportEnd}")
print("=" * 65)
print(f"  Monthly Spend  ({reportYear}-{reportMonth}):   ${total_this_month:>15,.2f}")
print(f"  {current_year} YTD Spend:            ${total_current_ytd:>15,.2f}")
print(f"  {prior_year} Full-Year Spend:       ${total_prior_full:>15,.2f}")
print(f"  YoY Change:                 ${yoy_change:>+15,.2f}  ({yoy_pct:+.1f}%)")
print("-" * 65)

top5_projects = monthly_df.groupby('project_name')['total_cost'].sum().nlargest(5)
print(f"\n  Top 5 Projects This Month:")
for proj, cost in top5_projects.items():
    pct = cost / total_this_month * 100
    print(f"    {proj:<42} ${cost:>10,.0f}  ({pct:.1f}%)")

top5_services = monthly_df.groupby('service_description')['total_cost'].sum().nlargest(5)
print(f"\n  Top 5 Services This Month:")
for svc, cost in top5_services.items():
    pct = cost / total_this_month * 100
    print(f"    {svc:<42} ${cost:>10,.0f}  ({pct:.1f}%)")
print("=" * 65)

  GCP FinOps Executive Summary
  Report Period : 2026-05-01  →  2026-06-01
  Monthly Spend  (2026-05):   $     511,368.47
  2026 YTD Spend:            $   2,053,581.56
  2025 Full-Year Spend:       $   2,223,845.57
  YoY Change:                 $    -170,264.01  (-7.7%)
-----------------------------------------------------------------

  Top 5 Projects This Month:
    prj-dac-aces-ops                           $    80,434  (15.7%)
    wo-wos-reporting-prod                      $    70,260  (13.7%)
    prj-edp-prod-edw                           $    36,250  (7.1%)
    prj-ai-traffic-dev                         $    15,243  (3.0%)
    prj-initiatives-nonprod                    $    13,062  (2.6%)

  Top 5 Services This Month:
    Workday Inc on Google                      $   143,104  (28.0%)
    BigQuery                                   $    71,967  (14.1%)
    Support                                    $    40,854  (8.0%)
    App Engine                                 $    38,299  (7.

## Export — Interactive HTML Report for Leadership
Produces a single self-contained HTML file you can email or open in a browser.

In [18]:
from datetime import datetime as dt

charts = [
    (fig_proj,  "Top 20 Projects — This Month"),
    (fig_svc,   "Top 15 Services — This Month"),
    (fig_tree,  "Service Cost Breakdown (Treemap)"),
    (fig_trend, "Monthly Spend Trend by Year"),
    (fig_year,  "Annual Spend with YoY Change"),
    (fig_exec,  f"Executive Dashboard: {prior_year} vs {current_year}"),
]

yoy_color = '#d32f2f' if yoy_change > 0 else '#388e3c'

html_parts = [f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>GCP FinOps Report — {reportYear}-{reportMonth}</title>
  <style>
    * {{ box-sizing: border-box; }}
    body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 0; padding: 0;
           background: #f0f2f5; color: #333; }}
    header {{ background: #1a73e8; color: white; padding: 24px 40px; }}
    header h1 {{ margin: 0; font-size: 28px; }}
    header p  {{ margin: 6px 0 0; opacity: 0.85; font-size: 14px; }}
    .kpi-row {{ display: flex; gap: 20px; padding: 24px 40px; flex-wrap: wrap; }}
    .kpi {{ background: white; border-radius: 10px; padding: 20px 28px;
            flex: 1; min-width: 180px; box-shadow: 0 2px 6px rgba(0,0,0,.1); }}
    .kpi-value {{ font-size: 30px; font-weight: 700; color: #1a73e8; }}
    .kpi-label {{ font-size: 13px; color: #888; margin-top: 4px; }}
    .kpi-value.red {{ color: #d32f2f; }}
    .kpi-value.green {{ color: #388e3c; }}
    .section {{ background: white; margin: 0 40px 28px; border-radius: 10px;
               padding: 24px; box-shadow: 0 2px 6px rgba(0,0,0,.1); }}
    .section h2 {{ margin: 0 0 16px; font-size: 18px; color: #1a73e8;
                  border-bottom: 2px solid #e8f0fe; padding-bottom: 10px; }}
    footer {{ text-align: center; padding: 20px; color: #999; font-size: 12px; }}
  </style>
</head>
<body>
<header>
  <h1>GCP FinOps Report</h1>
  <p>Period: {reportStart} &ndash; {reportEnd} &nbsp;|&nbsp; Generated: {dt.now().strftime('%Y-%m-%d %H:%M')}</p>
</header>

<div class="kpi-row">
  <div class="kpi">
    <div class="kpi-value">${total_this_month:,.0f}</div>
    <div class="kpi-label">Monthly Spend ({reportYear}-{reportMonth})</div>
  </div>
  <div class="kpi">
    <div class="kpi-value">${total_current_ytd:,.0f}</div>
    <div class="kpi-label">{current_year} YTD Spend</div>
  </div>
  <div class="kpi">
    <div class="kpi-value">${total_prior_full:,.0f}</div>
    <div class="kpi-label">{prior_year} Full-Year Spend</div>
  </div>
  <div class="kpi">
    <div class="kpi-value {'red' if yoy_change > 0 else 'green'}">{yoy_pct:+.1f}%</div>
    <div class="kpi-label">YoY Change (${yoy_change:+,.0f})</div>
  </div>
</div>
"""]

for fig, title in charts:
    chart_html = pio.to_html(fig, include_plotlyjs='cdn', full_html=False)
    html_parts.append(
        f'<div class="section"><h2>{title}</h2>{chart_html}</div>'
    )

html_parts.append(
    f'<footer>GCP FinOps Report &mdash; {reportYear}-{reportMonth} &mdash; Confidential</footer></body></html>'
)

report_filename = f"GCP_FinOps_Report_{reportYear}{reportMonth}.html"
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write('\n'.join(html_parts))

print(f"Report saved: {report_filename}")

# Also save aggregated monthly CSV for records
csv_filename = f"GCP_Monthly_Cost_For_{reportMonth}_{reportYear}.csv"
monthly_df.to_csv(csv_filename, index=False)
print(f"Data CSV saved: {csv_filename}")

Report saved: GCP_FinOps_Report_202605.html
Data CSV saved: GCP_Monthly_Cost_For_05_2026.csv
